In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle
import warnings
warnings.filterwarnings('ignore')

In [23]:
df = pd.read_csv('Car Dataset Processed.csv')

print("Dataset shape:", df.shape)
print("\nDataset columns:", df.columns.tolist())
print("\nFirst few rows:")
df.head(2)
 

Dataset shape: (1499, 15)

Dataset columns: ['Unnamed: 0', 'car_name', 'registration_year', 'insurance_validity', 'fuel_type', 'seats', 'kms_driven', 'ownsership', 'transmission', 'manufacturing_year', 'mileage(kmpl)', 'engine(cc)', 'max_power(bhp)', 'torque(Nm)', 'price(in lakhs)']

First few rows:


,Unnamed: 0,car_name,registration_year,insurance_validity,fuel_type,seats,kms_driven,ownsership,transmission,manufacturing_year,mileage(kmpl),engine(cc),max_power(bhp),torque(Nm),price(in lakhs)
0,0,2017 Mercedes-Benz S-Class S400,Jul-17,Comprehensive,Petrol,5,56000,First Owner,Automatic,2017,7.81,2996.0,2996.0,333.0,63.75
1,1,2020 Nissan Magnite Turbo CVT XV Premium Opt BSVI,Jan-21,Comprehensive,Petrol,5,30615,First Owner,Automatic,2020,17.40,999.0,999.0,9863.0,8.99


In [24]:
# Encoding dictionaries
d1 = {'Comprehensive': 0, 'Third Party insurance': 1, 'Third Party': 1, 'Zero Dep': 2, 'Not Available': 3}
d2 = {'Petrol': 0, 'Diesel': 1, 'CNG': 2}
d3 = {'First Owner': 1, 'Second Owner': 2, 'Third Owner': 3, 'Fourth Owner': 4, 'Fifth Owner': 5}
d4 = {'Manual': 0, 'Automatic': 1}

In [25]:
# Apply encodings
df['insurance_validity'] = df['insurance_validity'].map(d1)
df['fuel_type'] = df['fuel_type'].map(d2)
df['ownsership'] = df['ownsership'].map(d3)
df['transmission'] = df['transmission'].map(d4)

In [26]:
# Add car age feature
df['manufacturing_year'] = pd.to_numeric(df['manufacturing_year'], errors='coerce')
df['car_age'] = 2024 - df['manufacturing_year']

In [27]:
# Feature engineering
X = df[['insurance_validity', 'fuel_type', 'kms_driven', 'ownsership', 'transmission', 
         'mileage(kmpl)', 'engine(cc)', 'max_power(bhp)', 'torque(Nm)', 'car_age', 'seats']].copy()

In [28]:
# Handle missing values
X = X.fillna(X.median())
 
Y = df['price(in lakhs)']

In [29]:
# Remove outliers using IQR
Q1 = Y.quantile(0.25)
Q3 = Y.quantile(0.75)
IQR = Q3 - Q1
mask = (Y >= Q1 - 1.5*IQR) & (Y <= Q3 + 1.5*IQR)
X = X[mask]
Y = Y[mask]

print(f"\nDataset after removing outliers: {len(X)} samples")



Dataset after removing outliers: 1297 samples


In [30]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
 
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [31]:
# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'KNN-5': KNeighborsRegressor(n_neighbors=5),
    'KNN-7': KNeighborsRegressor(n_neighbors=7),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=15),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, learning_rate=0.1),
    'SVM-RBF': SVR(kernel='rbf', C=100, gamma='scale'),
}
 
print("\n" + "="*70)
print("MODEL PERFORMANCE COMPARISON")
print("="*70)

results = {}
for name, model in models.items():
    if name in ['Random Forest', 'Gradient Boosting', 'Linear Regression', 'Ridge Regression']:
        # These models work well with unscaled features
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        # SVM and KNN benefit from scaling
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    
    print(f"\n{name}:")
    print(f"  R² Score:  {r2:.4f}")
    print(f"  RMSE:      {rmse:.4f} lakhs")
    print(f"  MAE:       {mae:.4f} lakhs")



MODEL PERFORMANCE COMPARISON

Linear Regression:
  R² Score:  0.4549
  RMSE:      5.0601 lakhs
  MAE:       3.2981 lakhs

Ridge Regression:
  R² Score:  0.4550
  RMSE:      5.0599 lakhs
  MAE:       3.2970 lakhs

KNN-5:
  R² Score:  0.5366
  RMSE:      4.6657 lakhs
  MAE:       2.7642 lakhs

KNN-7:
  R² Score:  0.5332
  RMSE:      4.6828 lakhs
  MAE:       2.8261 lakhs

Random Forest:
  R² Score:  0.8384
  RMSE:      2.7549 lakhs
  MAE:       1.4215 lakhs

Gradient Boosting:
  R² Score:  0.8523
  RMSE:      2.6336 lakhs
  MAE:       1.6993 lakhs

SVM-RBF:
  R² Score:  0.5131
  RMSE:      4.7823 lakhs
  MAE:       2.5113 lakhs


In [32]:
# Select best model (highest R²)
best_model_name = max(results, key=lambda x: results[x]['R2'])
print("\n" + "="*70)
print(f"BEST MODEL: {best_model_name}")
print(f"R² Score: {results[best_model_name]['R2']:.4f}")
print("="*70)
 
# Train final model on full data
final_model = models[best_model_name]
if best_model_name in ['Random Forest', 'Gradient Boosting', 'Linear Regression', 'Ridge Regression']:
    final_model.fit(X, Y)
else:
    X_scaled = scaler.fit_transform(X)
    final_model.fit(X_scaled, Y)



BEST MODEL: Gradient Boosting
R² Score: 0.8523


In [ ]:
# Save model and scaler
with open('model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
 
# with open('scaler.pkl', 'wb') as f:
#     pickle.dump(scaler, f)
 
# with open('model_info.pkl', 'wb') as f:
#     pickle.dump({
#         'model_name': best_model_name,
#         'feature_names': X.columns.tolist(),
#         'scaling_required': best_model_name in ['KNN-5', 'KNN-7', 'SVM-RBF'],
#         'encoding_dicts': {
#             'insurance_validity': d1,
#             'fuel_type': d2,
#             'ownsership': d3,
#             'transmission': d4
#         }
#     }, f)
 

In [36]:
# Example prediction
print("\n" + "="*70)
print("EXAMPLE PREDICTION")
print("="*70)
example = [[0, 0, 20000, 1, 0, 18.5, 1500, 120, 180, 5, 5]]
if best_model_name in ['Random Forest', 'Gradient Boosting', 'Linear Regression', 'Ridge Regression']:
    pred = final_model.predict(example)[0]
else:
    example_scaled = scaler.transform(example)
    pred = final_model.predict(example_scaled)[0]
 
print(f"Input: Insurance=Comprehensive, Fuel=Petrol, KMs=20000, Owner=First, Transmission=Manual")
print(f"Predicted Price: {pred:.2f} lakhs (₹{pred*100000:.0f})")
 


EXAMPLE PREDICTION
Input: Insurance=Comprehensive, Fuel=Petrol, KMs=20000, Owner=First, Transmission=Manual
Predicted Price: 9.84 lakhs (₹984072)
